# Kaggle Baby Steps: Train Remaining Components + MoE\n
\n
This notebook assumes **Component 1** and **Component 4** are already pretrained.\n
\n
It trains the remaining trainable parts in this order:\n
1. Component 2 routing head\n
2. MoE cache (for TBX11K + Montgomery + Shenzhen)\n
3. Expert pretraining (Component 5)\n
4. Joint MoE training (Components 3 + 5 + 6)\n
5. Boundary critic (Component 7)

## What To Upload To Kaggle\n
\n
Create a Kaggle Notebook and attach these datasets as inputs:\n
- Your project code as a Kaggle Dataset (zip of this repo)\n
- Montgomery dataset\n
- Shenzhen dataset\n
- TBX11K dataset\n
- MedSAM ViT-B checkpoint dataset (file: medsam_vit_b.pth)\n
- Optional: Component 1 adapter dataset (component1_adapters.safetensors)\n
- Optional: Component 4 decoder dataset (component4_mask_decoder.pt)

In [ ]:
import os\n
from pathlib import Path\n
\n
KAGGLE_INPUT = Path('/kaggle/input')\n
KAGGLE_WORKING = Path('/kaggle/working')\n
REPO = KAGGLE_WORKING / 'repo'\n
\n
print('Input root:', KAGGLE_INPUT)\n
print('Working root:', KAGGLE_WORKING)

In [ ]:
# Inspect mounted Kaggle datasets\n
for p in sorted(KAGGLE_INPUT.glob('*')):\n
    print(p)

In [ ]:
# Install minimal runtime dependencies\n
!pip -q install pyyaml safetensors tqdm scipy segment_anything

In [ ]:
# Unpack your repo dataset into /kaggle/working/repo\n
# Replace YOUR_REPO_DATASET with your Kaggle input dataset folder name.\n
!rm -rf /kaggle/working/repo\n
!mkdir -p /kaggle/working/repo\n
!cp -r /kaggle/input/YOUR_REPO_DATASET/* /kaggle/working/repo

In [ ]:
%cd /kaggle/working/repo\n
!python -V

In [ ]:
# Optional path overrides if your mounted folders are custom.\n
# Leave commented if auto-detection in kaggle_moe_train.py already finds them.\n
# os.environ['KAGGLE_MONTGOMERY_PATH'] = '/kaggle/input/your-montgomery/montgomery'\n
# os.environ['KAGGLE_SHENZHEN_PATH'] = '/kaggle/input/your-shenzhen/shenzhen'\n
# os.environ['KAGGLE_TBX11K_PATH'] = '/kaggle/input/your-tbx11k/TBX11K'\n
# os.environ['KAGGLE_MEDSAM_CKPT'] = '/kaggle/input/your-medsam/medsam_vit_b.pth'\n
# os.environ['KAGGLE_C1_ADAPTER'] = '/kaggle/input/your-c1/component1_adapters.safetensors'\n
# os.environ['KAGGLE_C4_DECODER'] = '/kaggle/input/your-c4/component4_mask_decoder.pt'

## Step 1: Train Component 2 Routing Head\n
\n
This uses your existing Component 1 data manifest logic and saves into checkpoints/component2 by default.

In [ ]:
!python -m src.training.train_component2_txv --config configs/component2_txv.yaml --paths configs/paths.yaml --component1_config configs/component1_dann.yaml

## Step 2-5: Train MoE Stack\n
\n
Run smoke first to verify setup, then run full.

In [ ]:
# Smoke run\n
!python scripts/kaggle_moe_train.py --mode smoke --phase all

In [ ]:
# Full run\n
!python scripts/kaggle_moe_train.py --mode full --phase all

## Save Outputs From Kaggle\n
\n
Trained artifacts are expected under /kaggle/working/moe_runs and /kaggle/working/moe_artifacts.\n
Zip them so they can be downloaded or saved as a Kaggle output dataset.

In [ ]:
!mkdir -p /kaggle/working/final_artifacts\n
!cp -r /kaggle/working/moe_runs /kaggle/working/final_artifacts/ || true\n
!cp -r /kaggle/working/moe_artifacts /kaggle/working/final_artifacts/ || true\n
!cp -r /kaggle/working/repo/checkpoints/component2 /kaggle/working/final_artifacts/component2 || true\n
!cd /kaggle/working; zip -r training_outputs.zip final_artifacts

In [ ]:
!ls -lah /kaggle/working | sed -n '1,120p'